In [2]:
from pathlib import Path

import cv2
import joblib
import numpy as np

from skimage.feature import local_binary_pattern

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)

PROJECT_DIR = Path(
    r"D:\proto\TextileDefectPrototype"
)

DATASET_DIR = PROJECT_DIR / "dataset"

NORMAL_DIR = DATASET_DIR / "normal"
DEFECT_DIR = DATASET_DIR / "defect"

MODEL_DIR = PROJECT_DIR / "models"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_PATH = (
    MODEL_DIR / "lbp_svm.pkl"
)

print("Project:", PROJECT_DIR)
print("Model:", MODEL_PATH)

Project: D:\proto\TextileDefectPrototype
Model: D:\proto\TextileDefectPrototype\models\lbp_svm.pkl


In [3]:
def extract_lbp_features(
    image_path,
    image_size=(128, 128),
    num_points=24,
    radius=3
):

    image = cv2.imread(
        str(image_path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:
        raise ValueError(
            f"Could not read image: {image_path}"
        )

    image = cv2.resize(
        image,
        image_size
    )

    lbp = local_binary_pattern(
        image,
        num_points,
        radius,
        method="uniform"
    )

    # Uniform LBP produces P + 2 bins
    num_bins = num_points + 2

    histogram, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(
            0,
            num_bins + 1
        ),
        range=(0, num_bins)
    )

    # Normalize histogram
    histogram = histogram.astype(
        np.float32
    )

    histogram /= (
        histogram.sum() + 1e-7
    )

    return histogram

In [6]:
X = []
y = []

# ==========================================
# NORMAL
# ==========================================

normal_files = [
    p for p in NORMAL_DIR.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg"}
]

print("Normal images:", len(normal_files))

for i, image_path in enumerate(normal_files):

    features = extract_lbp_features(image_path)

    X.append(features)
    y.append(0)

    if (i + 1) % 500 == 0:
        print(
            f"Processed normal: {i + 1}/{len(normal_files)}"
        )


# ==========================================
# DEFECT
# ==========================================

defect_files = [
    p for p in DEFECT_DIR.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg"}
]

print("Defect images:", len(defect_files))

for i, image_path in enumerate(defect_files):

    features = extract_lbp_features(image_path)

    X.append(features)
    y.append(1)

    if (i + 100) % 100 == 0:
        print(
            f"Processed defect: {i + 1}/{len(defect_files)}"
        )


# ==========================================
# CONVERT TO NUMPY
# ==========================================

X = np.array(X)
y = np.array(y)

print()
print("Feature extraction complete.")
print("Feature shape:", X.shape)
print("Labels shape:", y.shape)
print("Normal samples:", np.sum(y == 0))
print("Defect samples:", np.sum(y == 1))

Normal images: 5794
Processed normal: 500/5794
Processed normal: 1000/5794
Processed normal: 1500/5794
Processed normal: 2000/5794
Processed normal: 2500/5794
Processed normal: 3000/5794
Processed normal: 3500/5794
Processed normal: 4000/5794
Processed normal: 4500/5794
Processed normal: 5000/5794
Processed normal: 5500/5794
Defect images: 590
Processed defect: 1/590
Processed defect: 101/590
Processed defect: 201/590
Processed defect: 301/590
Processed defect: 401/590
Processed defect: 501/590

Feature extraction complete.
Feature shape: (6384, 26)
Labels shape: (6384,)
Normal samples: 5794
Defect samples: 590


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 5107
Testing samples: 1277


In [8]:
svm_model = SVC(
    kernel="rbf",
    C=10,
    gamma="scale",
    class_weight="balanced",
    probability=True,
    random_state=42
)

svm_model.fit(
    X_train,
    y_train
)

print("LBP + SVM training complete.")

d:\proto\TextileDefectPrototype\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


LBP + SVM training complete.


In [9]:
y_pred = svm_model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "NORMAL",
            "DEFECT"
        ],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

Accuracy: 0.7291
Precision: 0.2121
Recall: 0.7119

Classification Report:
              precision    recall  f1-score   support

      NORMAL       0.96      0.73      0.83      1159
      DEFECT       0.21      0.71      0.33       118

    accuracy                           0.73      1277
   macro avg       0.59      0.72      0.58      1277
weighted avg       0.89      0.73      0.78      1277


Confusion Matrix:
[[847 312]
 [ 34  84]]


In [10]:
joblib.dump(
    svm_model,
    MODEL_PATH
)

print("LBP + SVM model saved:")
print(MODEL_PATH)

LBP + SVM model saved:
D:\proto\TextileDefectPrototype\models\lbp_svm.pkl


In [11]:
joblib.dump(
    svm_model,
    MODEL_PATH
)

print("LBP + SVM model saved:")
print(MODEL_PATH)

LBP + SVM model saved:
D:\proto\TextileDefectPrototype\models\lbp_svm.pkl
